# agentic_alpha_macro_credit_search_v1

Diagnostic R-G-E-V-D card for macro, credit, rate, dollar, and oil risk gates. It retrieves external knowledge from GitHub, papers, arXiv, and Twitter/X, builds a qlib provider from local macro inputs, reads the provider back through qlib `D.features`, evaluates macro credit spread and rate branches against `qlib_paper_sim_v1`, and records full evidence with `R.start`.

In [1]:
from pathlib import Path
import json
import sys
import shutil
import tempfile

import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R

CURRENT = Path.cwd().resolve()
ROOT = next(path for path in [CURRENT, *CURRENT.parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
sys.path.insert(0, str(ROOT))

from src.data.qlib_native import QlibProviderConfig, build_qlib_store_from_frame, load_qlib_feature_frame
from src.factor import build_macro_credit_frame, compact_macro_credit_summary, evaluate_macro_credit_candidates
from src.workflow import load_agentic_alpha_backtest_metric

EXPERIMENT = 'agentic_alpha_macro_credit_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl_macro_credit_v1').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
PRODUCTION_CARD = ROOT / 'production' / 'qlib_paper_sim_v1'
RAW_SYMBOLS = ROOT / 'data' / 'raw' / 'symbols'
RAW_EXTERNAL = ROOT / 'data' / 'raw' / 'external'
START = '2019-01-02'
END = '2026-06-09'
AS_OF = END
MAX_DRAWDOWN_FLOOR = -20.0
CARD.mkdir(parents=True, exist_ok=True)
baseline = load_agentic_alpha_backtest_metric(PRODUCTION_CARD, window='test')
baseline


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


AgenticAlphaBacktestMetric(name='qlib_paper_sim_v1', return_pct=6570.0232821043, max_drawdown_pct=-85.23168365, benchmark_return_pct=3530.8157772578, recorder_id='e0b5ed66c6514873b2fc79f2aec5874c')

In [2]:
def load_symbol(symbol: str) -> pd.DataFrame:
    frame = pd.read_csv(RAW_SYMBOLS / f'{symbol}.csv')
    frame['datetime'] = pd.to_datetime(frame['date']).dt.normalize()
    return frame[['datetime', 'close', 'volume']].sort_values('datetime').drop_duplicates('datetime')

def load_observation(file_name: str, value_col: str | None, output_col: str) -> pd.Series:
    frame = pd.read_csv(RAW_EXTERNAL / file_name)
    date_col = [col for col in frame.columns if col.lower() in {'date', 'datetime', 'observation_date'}][0]
    if value_col is None:
        value_col = [col for col in frame.columns if col != date_col][-1]
    normalized = frame[[date_col, value_col]].copy()
    normalized['datetime'] = pd.to_datetime(normalized[date_col], errors='coerce').dt.normalize()
    normalized[output_col] = pd.to_numeric(normalized[value_col], errors='coerce')
    normalized = normalized.dropna(subset=['datetime']).sort_values('datetime')
    return normalized.groupby('datetime')[output_col].last()

soxl = load_symbol('SOXL').rename(columns={'close': 'soxl_close', 'volume': 'soxl_volume'})
soxx = load_symbol('SOXX').rename(columns={'close': 'soxx_close', 'volume': 'soxx_volume'})
store_frame = soxl.merge(soxx, on='datetime', how='inner')
store_frame = store_frame.loc[(store_frame['datetime'] >= pd.Timestamp(START)) & (store_frame['datetime'] <= pd.Timestamp(END))].copy()
calendar = pd.DatetimeIndex(store_frame['datetime'])
external_specs = {
    'yc10y2y': ('yield_curve_10y2y.csv', 'value'),
    'yc10y3m': ('yield_curve_10y3m.csv', 'value'),
    'credit_baa': ('credit_spread_baa.csv', 'credit_spread'),
    'baa_yield': ('baa_yield.csv', 'BAA'),
    'hy_spread': ('credit_spread_hy.csv', None),
    'dxy': ('dxy_broad.csv', 'value'),
    'oil': ('oil_wti.csv', 'value'),
}
for output_col, (file_name, value_col) in external_specs.items():
    source_path = RAW_EXTERNAL / file_name
    if source_path.exists():
        store_frame[output_col] = load_observation(file_name, value_col, output_col).reindex(calendar, method='ffill').to_numpy()
store_frame['instrument'] = 'SOXL'
store_frame = store_frame[['instrument', 'datetime', 'soxl_close', 'soxx_close', 'yc10y2y', 'yc10y3m', 'credit_baa', 'baa_yield', 'hy_spread', 'dxy', 'oil']]
store_frame.tail()


,instrument,datetime,soxl_close,soxx_close,yc10y2y,yc10y3m,credit_baa,baa_yield,hy_spread,dxy,oil
4082,SOXL,2026-06-03,280.54,615.68,0.57,0.67,1.81,6.04,2.86,118.7294,91.06
4083,SOXL,2026-06-04,262.70,602.72,0.57,0.67,1.81,6.04,2.86,118.7294,91.06
4084,SOXL,2026-06-05,182.54,539.77,0.57,0.67,1.81,6.04,2.86,118.7294,91.06
4085,SOXL,2026-06-08,211.44,571.45,0.57,0.67,1.81,6.04,2.86,118.7294,91.06
4086,SOXL,2026-06-09,201.68,562.14,0.57,0.67,1.81,6.04,2.86,118.7294,91.06


In [3]:
provider_manifest = build_qlib_store_from_frame(
    store_frame,
    provider_uri=PROVIDER_URI,
    market='all',
    region='us',
    as_of=AS_OF,
    source='data/raw/symbols/SOXL,SOXX + data/raw/external/yield_curve,credit,dxy,oil',
)
shutil.copyfile(PROVIDER_URI / 'qlib_store_manifest.json', CARD / 'qlib_store_manifest.json')
provider_manifest


[55780:MainThread](2026-06-23 19:25:49,548) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[55780:MainThread](2026-06-23 19:25:49,550) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[55780:MainThread](2026-06-23 19:25:49,550) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_macro_credit_v1')}


[55780:MainThread](2026-06-23 19:25:49,557) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[55780:MainThread](2026-06-23 19:25:49,557) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[55780:MainThread](2026-06-23 19:25:49,558) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_macro_credit_v1')}


[55780:MainThread](2026-06-23 19:25:49,563) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[55780:MainThread](2026-06-23 19:25:49,564) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[55780:MainThread](2026-06-23 19:25:49,564) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_macro_credit_v1')}


{'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxl_macro_credit_v1',
 'market': 'all',
 'region': 'us',
 'freq': 'day',
 'as_of': '2026-06-09',
 'source': 'data/raw/symbols/SOXL,SOXX + data/raw/external/yield_curve,credit,dxy,oil',
 'instruments': ['SOXL'],
 'fields': ['$soxl_close',
  '$soxx_close',
  '$yc10y2y',
  '$yc10y3m',
  '$credit_baa',
  '$baa_yield',
  '$hy_spread',
  '$dxy',
  '$oil'],
 'calendar_start': '2019-01-02',
 'calendar_end': '2026-06-09',
 'row_count': 1869,
 'feature_file_count': 9,
 'audit_snapshot_hash': '547b3f7225190317',
 'operation': 'qlib_store_build',
 'status': 'ok',
 'hash': 'c70954c93801432c',
 'source_api': ['qlib.init',
  'qlib.data.storage.file_storage.FileFeatureStorage',
  'qlib.data.D.calendar',
  'qlib.data.D.instruments',
  'qlib.data.D.features']}

In [4]:
config = QlibProviderConfig(provider_uri=PROVIDER_URI, region='us', market='all', freq='day')
qlib_frame = load_qlib_feature_frame(
    config,
    instruments=['SOXL'],
    fields=provider_manifest['fields'],
    start=START,
    end=END,
)
feature_frame = build_macro_credit_frame(qlib_frame, start=START, end=END, min_history=20)
feature_frame[['datetime', 'next_return', 'soxx_trend20', 'macro_risk_score20', 'soxl_hv20']].head()


[55780:MainThread](2026-06-23 19:25:49,581) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[55780:MainThread](2026-06-23 19:25:49,581) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[55780:MainThread](2026-06-23 19:25:49,582) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_macro_credit_v1')}


,datetime,next_return,soxx_trend20,macro_risk_score20,soxl_hv20
0,2019-01-31,0.034953,0.093528,0.25,1.252773
1,2019-02-01,0.015070,0.175173,1.25,1.028555
2,2019-02-04,0.013952,0.130099,1.25,0.951454
3,2019-02-05,0.080356,0.114663,1.00,0.939829
4,2019-02-06,-0.066786,0.149683,1.00,0.957823


In [5]:
external_sources = pd.DataFrame(
    [
        {
            'source_type': 'github',
            'title': 'agentic-alpha R-G-E-V-D workflow',
            'url': 'https://github.com/JacobDu/agentic-alpha',
            'extracted_signal': 'Use retrieve-generate-evaluate-validate-distill loops to close macro and credit risk branches with comparable evidence.',
            'tags': 'workflow_reference',
        },
        {
            'source_type': 'github',
            'title': 'tactical-asset-allocation',
            'url': 'https://github.com/oronimbus/tactical-asset-allocation',
            'extracted_signal': 'Tactical allocation programs motivate explicit macro state and trend filters instead of a single static SOXL exposure.',
            'tags': 'taa_macro',
        },
        {
            'source_type': 'paper',
            'title': 'A Quantitative Approach to Tactical Asset Allocation',
            'url': 'https://mebfaber.com/wp-content/uploads/2016/05/SSRN-id962461.pdf',
            'extracted_signal': 'Trend filters can manage large equity drawdowns; test whether macro credit filters improve SOXL risk control.',
            'tags': 'trend_following',
        },
        {
            'source_type': 'paper',
            'title': 'Leverage for the Long Run',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2741701',
            'extracted_signal': 'Volatility is the enemy of leverage and risk should be reduced when leverage conditions deteriorate.',
            'tags': 'leverage_risk',
        },
        {
            'source_type': 'paper',
            'title': 'Leveraged ETFs in Asset Allocation: Opportunity or Trap?',
            'url': 'https://quantpedia.com/leveraged-etfs-in-asset-allocation-opportunity-or-trap/',
            'extracted_signal': 'Leveraged exposure needs active trend and risk management; test macro credit spread gates against SOXL drawdown.',
            'tags': 'leveraged_etf',
        },
        {
            'source_type': 'arxiv',
            'title': 'Compounding Effects in Leveraged ETFs',
            'url': 'https://arxiv.org/html/2504.20116v1',
            'extracted_signal': 'Leveraged ETF compounding benefits from trending regimes and suffers in volatile regimes.',
            'tags': 'letf_compounding',
        },
        {
            'source_type': 'twitter',
            'title': 'TrendSpider SOXL volatility seed',
            'url': 'https://x.com/TrendSpider/status/2063063329531109596',
            'extracted_signal': 'Treat SOXL as explicitly volatility-sensitive and require drawdown gates for any high-return candidate.',
            'tags': 'soxl_volatility_seed',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)
external_sources


,source_type,title,url,extracted_signal,tags
0,github,agentic-alpha R-G-E-V-D workflow,https://github.com/JacobDu/agentic-alpha,Use retrieve-generate-evaluate-validate-distil...,workflow_reference
1,github,tactical-asset-allocation,https://github.com/oronimbus/tactical-asset-al...,Tactical allocation programs motivate explicit...,taa_macro
2,paper,A Quantitative Approach to Tactical Asset Allo...,https://mebfaber.com/wp-content/uploads/2016/0...,Trend filters can manage large equity drawdown...,trend_following
3,paper,Leverage for the Long Run,https://papers.ssrn.com/sol3/papers.cfm?abstra...,Volatility is the enemy of leverage and risk s...,leverage_risk
4,paper,Leveraged ETFs in Asset Allocation: Opportunit...,https://quantpedia.com/leveraged-etfs-in-asset...,Leveraged exposure needs active trend and risk...,leveraged_etf
5,arxiv,Compounding Effects in Leveraged ETFs,https://arxiv.org/html/2504.20116v1,Leveraged ETF compounding benefits from trendi...,letf_compounding
6,twitter,TrendSpider SOXL volatility seed,https://x.com/TrendSpider/status/2063063329531...,Treat SOXL as explicitly volatility-sensitive ...,soxl_volatility_seed


In [6]:
workflow_yaml = f'''experiment: {EXPERIMENT}
route: research
provider_uri: {PROVIDER_URI.as_posix()}
recorder_uri: {RECORDER_URI.as_posix()}
market: all
region: us
data_range:
  start: {START}
  end: {END}
agentic_alpha:
  source: https://github.com/JacobDu/agentic-alpha
  stage_order:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  require_return_above_baseline: true
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
artifact_policy:
  policy: slim_card_recorder_first
  workflow_config: workflow.yaml
  recorder_only_artifacts:
    - macro_credit_full_candidate_grid.csv
    - predictions
    - signals
    - portfolio_report
    - positions
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')
print(workflow_yaml)


experiment: agentic_alpha_macro_credit_search_v1
route: research
provider_uri: /Users/rziiiii/dev/workspace/data/qlib/soxl_macro_credit_v1
recorder_uri: /Users/rziiiii/dev/workspace/research/agentic_alpha_macro_credit_search_v1/qlib_recorder
market: all
region: us
data_range:
  start: 2019-01-02
  end: 2026-06-09
agentic_alpha:
  source: https://github.com/JacobDu/agentic-alpha
  stage_order:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  require_return_above_baseline: true
  max_drawdown_floor: -20.0
artifact_policy:
  policy: slim_card_recorder_first
  workflow_config: workflow.yaml
  recorder_only_artifacts:
    - macro_credit_full_candidate_grid.csv
    - predictions
    - signals
    - portfolio_report
    - positions



In [7]:
candidate_metrics_full = evaluate_macro_credit_candidates(
    feature_frame,
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
)
search_summary = compact_macro_credit_summary(
    candidate_metrics_full,
    branches=sorted(candidate_metrics_full['branch'].unique()),
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
)
compact_rows = []
for branch, rows in candidate_metrics_full.groupby('branch'):
    compact_rows.append(rows.sort_values('return_pct', ascending=False).iloc[0])
    drawdown_rows = rows.loc[rows['passes_drawdown_gate']].sort_values('return_pct', ascending=False)
    if not drawdown_rows.empty:
        compact_rows.append(drawdown_rows.iloc[0])
candidate_metrics = pd.DataFrame(compact_rows).drop_duplicates('candidate').sort_values(['branch', 'return_pct'], ascending=[True, False])
candidate_metrics.to_csv(CARD / 'agentic_alpha_macro_credit_candidate_metrics.csv', index=False)
(CARD / 'agentic_alpha_macro_credit_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
failure_modes = (
    '# Failure Modes\n'
    '- high-return macro_credit_rate_gate candidates still carry SOXL-like crash drawdowns.\n'
    '- drawdown-gated macro_composite_throttle candidates reduce risk but miss the production baseline return gate.\n'
    '- trend_drawdown_stop is a control branch and did not create a drawdown-safe high-return candidate.\n'
)
(CARD / 'failure_modes.md').write_text(failure_modes, encoding='utf-8')
candidate_metrics[['branch', 'candidate', 'return_pct', 'max_drawdown_pct', 'beats_baseline_return', 'passes_drawdown_gate']]


,branch,candidate,return_pct,max_drawdown_pct,beats_baseline_return,passes_drawdown_gate
3139,macro_composite_throttle,risklb20_trend50_b1_tw0.75_rw1.5_volNone,48364.949428,-80.054981,True,False
12016,macro_composite_throttle,risklb63_trend100_b0.5_tw0.75_rw1.5_vol0.4,762.191118,-19.034843,False,True
526,macro_credit_rate_gate,risklb42_trend50_risk1_scale2.5,83085.316944,-80.344651,True,False
56,macro_credit_rate_gate,risklb20_trend50_risk0_scale0.5,392.587701,-17.951426,False,True
22555,trend_drawdown_stop,lb126_stop-0.3_re0.02_scale1.25,1597.849606,-61.013320,False,False


In [8]:
R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder.log_params(
        workflow='agentic_alpha_macro_credit_search',
        provider_uri=PROVIDER_URI.as_posix(),
        baseline_return_pct=baseline.return_pct,
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        source_count=len(external_sources),
    )
    recorder.log_metrics(
        passes_both_count=float(search_summary['passes_both_count']),
        best_macro_credit_rate_gate_return=float(search_summary['branch_results']['macro_credit_rate_gate']['best_return_pct']),
        best_macro_credit_rate_gate_drawdown=float(search_summary['branch_results']['macro_credit_rate_gate']['best_return_max_drawdown_pct']),
        best_drawdown_gated_return=float(search_summary['branch_results']['macro_composite_throttle']['best_drawdown_gate_return_pct']),
    )
    sidecar_paths = [
        CARD / 'agentic_alpha_macro_credit_candidate_metrics.csv',
        CARD / 'agentic_alpha_macro_credit_search_summary.json',
        CARD / 'external_sources.csv',
        CARD / 'failure_modes.md',
        CARD / 'qlib_store_manifest.json',
    ]
    for path in sidecar_paths:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_macro_credit')
    with tempfile.TemporaryDirectory() as temp_dir:
        full_grid_path = Path(temp_dir) / 'macro_credit_full_candidate_grid.csv'
        candidate_metrics_full.to_csv(full_grid_path, index=False)
        recorder.log_artifact(full_grid_path.as_posix(), artifact_path='agentic_alpha_macro_credit')
    recorder_id = recorder.id
    recorder_local_dir = Path(recorder.get_local_dir())
recorder_id, recorder_local_dir


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[55780:MainThread](2026-06-23 19:25:51,409) INFO - qlib.workflow - [exp.py:258] - Experiment 693211767858480417 starts running ...


[55780:MainThread](2026-06-23 19:25:51,473) INFO - qlib.workflow - [recorder.py:345] - Recorder 6768729892374924a5da1ab1d7a3c0da starts running under Experiment 693211767858480417 ...


[55780:MainThread](2026-06-23 19:25:51,687) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


('6768729892374924a5da1ab1d7a3c0da',
 PosixPath('/Users/rziiiii/dev/workspace/research/agentic_alpha_macro_credit_search_v1/qlib_recorder/693211767858480417/6768729892374924a5da1ab1d7a3c0da'))

In [9]:
summary = {
    'experiment': EXPERIMENT,
    'card_type': 'diagnostic_search',
    'promotion_decision': 'not_promoted',
    'as_of': AS_OF,
    'data_range': [START, END],
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'market': 'all',
    'region': 'us',
    'baseline': {
        'name': 'qlib_paper_sim_v1',
        'return_pct': baseline.return_pct,
        'benchmark_return_pct': baseline.benchmark_return_pct,
        'max_drawdown_pct': baseline.max_drawdown_pct,
        'recorder_id': baseline.recorder_id,
    },
    'goal_gate': {
        'max_drawdown_floor': MAX_DRAWDOWN_FLOOR,
        'require_return_above_baseline': True,
    },
    'search_report': search_summary,
    'branch_results': search_summary['branch_results'],
    'macro_credit': {'selected_features': search_summary['selected_features']},
    'macro_credit_data': {
        'provider_manifest': 'data/qlib/soxl_macro_credit_v1/qlib_store_manifest.json',
        'card_provider_manifest': 'qlib_store_manifest.json',
        'source_fields': search_summary['selected_features'],
        'candidate_coverage_start': feature_frame['datetime'].min().date().isoformat(),
        'candidate_coverage_end': feature_frame['datetime'].max().date().isoformat(),
    },
    'external_source_counts': external_sources['source_type'].value_counts().sort_index().to_dict(),
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'model': 'https://qlib.readthedocs.io/en/latest/component/model.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
        'recorder': 'https://qlib.readthedocs.io/en/latest/component/recorder.html',
    },
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start', 'build_qlib_store_from_frame'],
    'workflow_config': 'workflow.yaml',
    'sidecars': {
        'candidate_metrics': 'agentic_alpha_macro_credit_candidate_metrics.csv',
        'external_sources': 'external_sources.csv',
        'failure_modes': 'failure_modes.md',
        'qlib_store_manifest': 'qlib_store_manifest.json',
        'search_summary': 'agentic_alpha_macro_credit_search_summary.json',
    },
    'artifact_policy': {
        'policy': 'slim_card_recorder_first',
        'workflow_config': 'workflow.yaml',
        'machine_entrypoint': 'summary.json',
        'primary_artifact': f'{EXPERIMENT}.ipynb',
        'kept_sidecars': [
            'agentic_alpha_macro_credit_candidate_metrics.csv',
            'agentic_alpha_macro_credit_search_summary.json',
            'external_sources.csv',
            'failure_modes.md',
            'qlib_store_manifest.json',
        ],
        'recorder_only_artifacts': {
            'macro_credit_full_candidate_grid': 'artifacts/agentic_alpha_macro_credit/macro_credit_full_candidate_grid.csv',
            'predictions': 'artifacts/pred.pkl',
            'labels': 'artifacts/label.pkl',
            'signals': 'artifacts/signals.pkl',
            'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
            'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
            'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
            'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        },
        'reproducible_from_recorder': True,
    },
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
summary


{'experiment': 'agentic_alpha_macro_credit_search_v1',
 'card_type': 'diagnostic_search',
 'promotion_decision': 'not_promoted',
 'as_of': '2026-06-09',
 'data_range': ['2019-01-02', '2026-06-09'],
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxl_macro_credit_v1',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/agentic_alpha_macro_credit_search_v1/qlib_recorder',
 'recorder_id': '6768729892374924a5da1ab1d7a3c0da',
 'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/agentic_alpha_macro_credit_search_v1/qlib_recorder/693211767858480417/6768729892374924a5da1ab1d7a3c0da',
 'market': 'all',
 'region': 'us',
 'baseline': {'name': 'qlib_paper_sim_v1',
  'return_pct': 6570.0232821043,
  'benchmark_return_pct': 3530.8157772578,
  'max_drawdown_pct': -85.23168365,
  'recorder_id': 'e0b5ed66c6514873b2fc79f2aec5874c'},
 'goal_gate': {'max_drawdown_floor': -20.0,
  'require_return_above_baseline': True},
 'search_report': {'workflow': 'agentic_alpha_macro_credit_searc

In [10]:
conclusion = (
    f'# 结论\n'
    f'- what_ran: 执行 {EXPERIMENT}，将 GitHub/论文/arXiv/Twitter 的 TAA、杠杆 ETF 和信用风险知识映射到收益率曲线、credit spread、BAA yield、DXY、oil 与 SOXX trend，并用 `build_qlib_store_from_frame` 构建 qlib provider 后通过 `D.features` 审计。\n'
    f'- key_metrics: best macro_credit_rate_gate return={summary["branch_results"]["macro_credit_rate_gate"]["best_return_pct"]:.4f}%, MaxDD={summary["branch_results"]["macro_credit_rate_gate"]["best_return_max_drawdown_pct"]:.4f}%；best drawdown-gated return={summary["branch_results"]["macro_composite_throttle"]["best_drawdown_gate_return_pct"]:.4f}%, MaxDD={summary["branch_results"]["macro_composite_throttle"]["best_drawdown_gate_max_drawdown_pct"]:.4f}%。\n'
    f'- regime_window_differences: 信用/利率 gate 可筛掉部分风险期，但高收益候选仍继承 SOXL 崩盘回撤；vol-target throttle 能把 MaxDD 压到 20% 内，但收益远低于生产 baseline。\n'
    f'- promotion_decision: not_promoted；未找到同时满足收益超过现有生产策略且 MaxDD 小于 20% 的候选。\n'
    f'- risks: 宏观变量多为月度或外部日频序列，虽然用 t 日信号交易 t+1 收益，仍需 validated card 复核实时可得性和发布延迟。\n'
    f'- next_action: 继续 R-G-E-V-D，优先寻找能够在高收益候选上降低 crash gap 的非线性/组合保险分支。\n'
    f'- qlib_recorder_id: {recorder_id}\n'
)
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print(conclusion)


# 结论
- what_ran: 执行 agentic_alpha_macro_credit_search_v1，将 GitHub/论文/arXiv/Twitter 的 TAA、杠杆 ETF 和信用风险知识映射到收益率曲线、credit spread、BAA yield、DXY、oil 与 SOXX trend，并用 `build_qlib_store_from_frame` 构建 qlib provider 后通过 `D.features` 审计。
- key_metrics: best macro_credit_rate_gate return=83085.3169%, MaxDD=-80.3447%；best drawdown-gated return=762.1911%, MaxDD=-19.0348%。
- regime_window_differences: 信用/利率 gate 可筛掉部分风险期，但高收益候选仍继承 SOXL 崩盘回撤；vol-target throttle 能把 MaxDD 压到 20% 内，但收益远低于生产 baseline。
- promotion_decision: not_promoted；未找到同时满足收益超过现有生产策略且 MaxDD 小于 20% 的候选。
- risks: 宏观变量多为月度或外部日频序列，虽然用 t 日信号交易 t+1 收益，仍需 validated card 复核实时可得性和发布延迟。
- next_action: 继续 R-G-E-V-D，优先寻找能够在高收益候选上降低 crash gap 的非线性/组合保险分支。
- qlib_recorder_id: 6768729892374924a5da1ab1d7a3c0da

